In [1]:
from pathlib import Path
import time
import gc

import pandas as pd
import cudf
import cupy as cp

DATA_DIR = Path.home() / "datasets" / "nyc-taxi" / "2025" / "cleaned"

REPORT_DIR = (
    Path.home()
    / "projects"
    / "portfolio"
    / "project-00-spark-validation"
    / "reports"
)

REPORT_DIR.mkdir(exist_ok=True)

files = sorted(
    DATA_DIR.glob("yellow_tripdata_2025-??_clean.parquet")
)

tiers = {
    "1_month": files[:1],
    "3_months": files[:3],
    "6_months": files[:6],
    "12_months": files[:12],
}

timing_cols = [
    "load",
    "filter",
    "null_count",
    "groupby",
    "feature_engineering",
    "sort",
    "total",
]

print("Files found:", len(files))

Files found: 12


In [2]:
def timed_gpu(func):
    cp.cuda.Stream.null.synchronize()

    start = time.perf_counter()

    result = func()

    cp.cuda.Stream.null.synchronize()

    elapsed = time.perf_counter() - start

    return result, elapsed

In [3]:
def benchmark_cudf(file_list):
    results = {}

    # 1. Load
    df, results["load"] = timed_gpu(
        lambda: cudf.concat(
            [cudf.read_parquet(str(f)) for f in file_list],
            ignore_index=True
        )
    )

    # 2. Filter
    filtered, results["filter"] = timed_gpu(
        lambda: df[
            (df["trip_distance"] > 0)
            & (df["fare_amount"] > 0)
            & (df["total_amount"] > 0)
        ]
    )

    # 3. Null count
    null_counts, results["null_count"] = timed_gpu(
        lambda: df.isna().sum()
    )

    # 4. Groupby
    grouped, results["groupby"] = timed_gpu(
        lambda: df.groupby("payment_type").agg({
            "VendorID": "count",
            "trip_distance": "mean",
            "fare_amount": "mean",
            "tip_amount": "mean",
            "total_amount": "mean",
        })
    )

    # 5. Feature engineering
    def make_features():
        temp = df.copy()

        temp["trip_duration_min"] = (
            temp["tpep_dropoff_datetime"]
            - temp["tpep_pickup_datetime"]
        ).dt.total_seconds() / 60

        temp["pickup_hour"] = (
            temp["tpep_pickup_datetime"].dt.hour
        )

        tip_pct = (
            temp["tip_amount"]
            / temp["fare_amount"]
            * 100
        )

        temp["tip_pct"] = tip_pct.where(
            temp["fare_amount"] > 0
        )

        return temp

    engineered, results["feature_engineering"] = timed_gpu(
        make_features
    )

    # 6. Sort
    sorted_df, results["sort"] = timed_gpu(
        lambda: df.sort_values(
            "total_amount",
            ascending=False
        )
    )

    results["rows"] = len(df)

    results["total"] = sum(
        results[key]
        for key in timing_cols[:-1]
    )

    return results

In [4]:
def run_repeated_gpu_benchmark(
    benchmark_func,
    tiers,
    backend="cuDF",
    runs=5
):
    records = []

    for tier_name, tier_files in tiers.items():
        print(f"\n{backend} — {tier_name}")

        for run_num in range(1, runs + 1):
            gc.collect()
            cp.get_default_memory_pool().free_all_blocks()

            result = benchmark_func(tier_files)

            result["tier"] = tier_name
            result["backend"] = backend
            result["run"] = run_num

            records.append(result)

            print(
                f"Run {run_num}: "
                f"{result['total']:.4f} s"
            )

    return pd.DataFrame(records)

In [5]:
# january single month test
test_gpu = benchmark_cudf(tiers["1_month"])
test_gpu

{'load': 0.6031001799856313,
 'filter': 0.047246425005141646,
 'null_count': 0.043586393003351986,
 'groupby': 0.02747585199540481,
 'feature_engineering': 0.2051790570258163,
 'sort': 0.0731417799834162,
 'rows': 3475082,
 'total': 0.9997296869987622}

In [6]:
# row confirm
test_gpu["rows"]

3475082

In [7]:
# benchmark 5 rep
cudf_runs = run_repeated_gpu_benchmark(
    benchmark_cudf,
    tiers,
    backend="cuDF",
    runs=5
)

cudf_runs


cuDF — 1_month
Run 1: 0.3093 s
Run 2: 0.2705 s
Run 3: 0.2729 s
Run 4: 0.2728 s
Run 5: 0.2713 s

cuDF — 3_months
Run 1: 0.7893 s
Run 2: 0.7926 s
Run 3: 0.7928 s
Run 4: 0.8012 s
Run 5: 0.7902 s

cuDF — 6_months
Run 1: 1.6364 s
Run 2: 1.6384 s
Run 3: 1.6353 s
Run 4: 1.6566 s
Run 5: 1.6398 s

cuDF — 12_months
Run 1: 3.2789 s
Run 2: 3.2888 s
Run 3: 3.2558 s
Run 4: 3.2558 s
Run 5: 3.2598 s


,load,filter,null_count,groupby,feature_engineering,sort,rows,total,tier,backend,run
0,0.119965,0.035203,0.022187,0.009228,0.059335,0.063402,3475082,0.309319,1_month,cuDF,1
1,0.085836,0.034905,0.021809,0.006226,0.058759,0.062990,3475082,0.270526,1_month,cuDF,2
2,0.085972,0.034844,0.021708,0.008354,0.058896,0.063107,3475082,0.272881,1_month,cuDF,3
3,0.085750,0.035171,0.021837,0.008187,0.058900,0.062981,3475082,0.272826,1_month,cuDF,4
4,0.085599,0.034789,0.021770,0.006846,0.059303,0.062951,3475082,0.271258,1_month,cuDF,5
5,0.260260,0.098633,0.046307,0.015872,0.158173,0.210024,11197681,0.789268,3_months,cuDF,1
6,0.261311,0.098757,0.046149,0.015912,0.158309,0.212123,11197681,0.792562,3_months,cuDF,2
7,0.261370,0.099755,0.046204,0.015930,0.157727,0.211778,11197681,0.792764,3_months,cuDF,3
8,0.262805,0.098912,0.046288,0.024284,0.158201,0.210733,11197681,0.801222,3_months,cuDF,4
9,0.261881,0.098597,0.046156,0.015917,0.157253,0.210405,11197681,0.790208,3_months,cuDF,5


In [8]:
cudf_runs.shape

(20, 11)

In [9]:
cudf_runs.to_csv(
    REPORT_DIR / "cudf_benchmark_runs.csv",
    index=False
)

In [10]:
cudf_summary = (
    cudf_runs
    .groupby("tier")[timing_cols]
    .agg(["median", "mean", "std"])
)

cudf_summary

load                        filter                      \
             median      mean       std    median      mean       std   
tier                                                                    
12_months  1.091773  1.091462  0.001884  0.393350  0.393432  0.000681   
1_month    0.085836  0.092624  0.015284  0.034905  0.034982  0.000192   
3_months   0.261370  0.261525  0.000927  0.098757  0.098931  0.000477   
6_months   0.550388  0.551151  0.001645  0.200331  0.200065  0.000721   

          null_count                       groupby  ...            \
              median      mean       std    median  ...       std   
tier                                                ...             
12_months   0.137593  0.137584  0.000533  0.062991  ...  0.013935   
1_month     0.021809  0.021862  0.000188  0.008187  ...  0.001212   
3_months    0.046204  0.046221  0.000074  0.015917  ...  0.003746   
6_months    0.077854  0.077919  0.000350  0.031758  ...  0.010923   

          feature_engineering                          sort            \
                       median      mean       std    median      mean   
tier                                                                    
12_months            0.633408  0.632976  0.003410  0.939807  0.939937   
1_month              0.058900  0.059038  0.000262  0.062990  0.063086   
3_months             0.158173  0.157933  0.000440  0.210733  0.211012   
6_months             0.317399  0.317900  0.001348  0.457390  0.457720   

                        total                      
                std    median      mean       std  
tier                                               
12_months  0.001313  3.259784  3.267826  0.015166  
1_month    0.000186  0.272826  0.279362  0.016777  
3_months   0.000901  0.792562  0.793205  0.004726  
6_months   0.000842  1.638407  1.641324  0.008736  

[4 rows x 21 columns]

In [11]:
cudf_summary.to_csv(
    REPORT_DIR / "cudf_benchmark_summary.csv"
)